# Preprocessing Pipeline for Fraud Detection Datasets (Batch & Real‑Time Ready)

## Overview
This notebook preprocesses three fraud‑detection datasets (Sparkov, IEEE‑CIS, European Credit Card Holders 2013) **once** and outputs:

1. **Raw validation & test CSVs** – exactly as they appeared right after the temporal split, with **no further transformations**, and **a standardised target column `Fraud`**.
2. **Resampled training CSVs** – one for each combination of fast feature‑selection method and sampling technique, ready to train models. The target column is also `Fraud`.
3. **Serialised preprocessing state** (`joblib` files) – every fitted object is saved so that the exact same transformations can be applied to new, raw transactions without re‑running the notebook.

**Goal:** After this notebook, train models on the resampled CSVs. For real‑world inference:
- Load a raw transaction (or `_val.csv` / `_test.csv`).
- Load `_base_state.joblib` and the chosen `combo_state.joblib`.
- Apply imputation, feature engineering, frequency encoding, scaling, and feature selection step‑by‑step, measuring each step’s execution time.
- Feed the final vector to the trained model and obtain a fraud / legitimate decision.

## Datasets
| Dataset            | Type      | Original Target | Standardised Target | Time Column    | Source Files / Pattern                                  |
|--------------------|-----------|-----------------|---------------------|----------------|---------------------------------------------------------|
| Sparkov            | Synthetic | `is_fraud`      | `Fraud`             | `unix_time`    | `*.csv` (multiple files concatenated)                   |
| IEEE‑CIS           | Real      | `isFraud`       | `Fraud`             | `TransactionDT`| `train_transaction.csv` + `train_identity.csv` (merged) |
| EuropeanCard       | Real      | `Class`         | `Fraud`             | `Time`         | `creditcard.csv` (V1‑V28, Time, Amount)                |

**Test run**: 1% random sample (`TEST_FRACTION = 0.01`). Remove `df.sample(...)` for the full dataset.

## Pipeline Steps (applied per dataset)

### 0. Target Standardisation
**Immediately after loading**, the dataset‑specific fraud column is renamed to `Fraud`. This ensures all downstream outputs and scripts use a single label name, avoiding bugs in evaluation loops.

### 1. Temporal Split
Data sorted by time, split forward:
- **60%** earliest → training
- **20%** middle → validation
- **20%** most recent → test

The raw validation and test sets (with the target `Fraud`) are saved as `{dataset}_val.csv` and `{dataset}_test.csv` – ready for inference benchmarking.

### 2. Missing Value Handling
- Drop columns with >5% missing (training set).
- Impute categoricals (mode) and numericals (median). Fitted imputers saved.

### 3. Feature Engineering
Training‑set aggregations only, merged to validation/test. Saved tables handle unseen entities with fallback values.

- **Sparkov**: Datetime decomposition, log amount, per‑card & per‑merchant stats, **vectorised Haversine distance** for speed.
- **IEEE‑CIS**: Synthetic datetime, log amount, per‑card (`card1`), per‑billing address (`addr1`), per‑email‑domain stats.
- **EuropeanCard**: Time features (hour, day, week), log amount. No aggregations (no card/account IDs).

### 4. High‑Cardinality Categoricals
Columns with >50 unique values → frequency‑encoded using training frequencies. Maps saved.

### 5. Encoding & Scaling
- Low‑cardinality categoricals: ordinal encoding (unknown → -1).
- Numerical features: standard scaling (zero mean, unit variance). Encoder and scaler saved.

### 6. Feature Selection (fast filters only)
- **SelectKBest**: ANOVA F‑test and Mutual Information, `k ∈ {5,10,15,20,30,50,all}` (k capped to actual feature count).
- **SelectPercentile** (ANOVA): 10%, 20%, 30%, 50%.
- **VarianceThreshold**: removes constant features.
- **Baseline**: `NoSelection_all`.

Selectors are saved in combo states.

### 7. Resampling
10 fast, scalable techniques applied after feature selection:

| Category      | Methods                                                                   |
|---------------|---------------------------------------------------------------------------|
| Oversampling  | SMOTE, BorderlineSMOTE, ADASYN                                            |
| Undersampling | RandomUnderSampler, ClusterCentroids, NearMiss (v3), TomekLinks, EditedNearestNeighbours |
| Hybrid        | SMOTE‑ENN, SMOTE‑Tomek                                                    |

Safe `k_neighbors` is adapted to minority class size.

### 8. Outputs & Serialisation

| Output | Path | Content |
|--------|------|---------|
| Raw validation | `../prepareddata/{dataset}_val.csv` | Original features + `Fraud` column, after temporal split |
| Raw test | `../prepareddata/{dataset}_test.csv` | Original features + `Fraud` column, after temporal split |
| Resampled training | `../prepareddata/{sampler}--{fs}--{dataset}--{timestamp}.csv` | Transformed, resampled training set with `Fraud` column |
| Base state | `../prepareddata/{dataset}_base_state.joblib` | Fitted imputers, encoder, scaler, aggregation tables, frequency maps |
| Combo state | `../prepareddata/{sampler}--{fs}--{dataset}--{timestamp}_state.joblib` | Feature selector and sampler |
| Processing log | `processing_times.csv` | Timing (selection, sampling, total) and class counts per file |

### 9. Downstream Workflow (our Plan)

1. **Model Training** – Pick a resampled CSV, train a classifier (XGBoost, etc.), save the model.
2. **Inference Benchmarking** – For each raw validation/test CSV:
   - Load CSV + `base_state.joblib` + `combo_state.joblib`.
   - Apply imputation, engineering, frequency encoding, scaling, selection step‑by‑step, timing each.
   - Predict with the saved model, measure total latency.
   - The target column is always `Fraud` – no renaming needed.

This design guarantees a clean separation of preprocessing, training, and inference, with full reproducibility and real‑world timing measurements.

In [1]:
import os
import math
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.feature_selection import (
    SelectKBest, f_classif, mutual_info_classif,
    SelectPercentile, VarianceThreshold
)

from imblearn.over_sampling import SMOTE,BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler, ClusterCentroids,NearMiss, TomekLinks,EditedNearestNeighbours
from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.over_sampling import ADASYN
import joblib
import time

warnings.filterwarnings('ignore')

In [2]:
# Global log for processing times and counts
processing_log = []   # will hold dicts for each successful combination

In [3]:
BASE_PATH = "../Dataset/"                     
DATASETS = {
    "Sparkov": {
        "folder": "SparkovGeneratedData",
        "files": "*.csv",          
        "target_col": "is_fraud",
        "time_col": "unix_time",
        "type": "Synthetic"
    },
    "IEEE-CIS": {
        "folder": "ieee-fraud-detection",
        "files": {
            "transaction": "train_transaction.csv",
            "identity": "train_identity.csv"
        },
        "target_col": "isFraud",
        "time_col": "TransactionDT",  # <-- time delta in IEEE-CIS
        "type": "Real"
    },
    "EuropeanCard": {
        "folder": ".",
        "files": "creditcard.csv",
        "target_col": "Class",
        "time_col": "Time",            # seconds since first transaction
        "type": "Real"
    }
}

TEST_FRACTION = 0.01   # 0.1% of the data
OUTPUT_DIR = "../prepareddata"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
def load_sparkov(folder, file_pattern, time_col):
    import glob
    files = glob.glob(os.path.join(BASE_PATH, folder, file_pattern))
    if not files:
        raise FileNotFoundError(f"No files found in {folder}")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    return df.sort_values(time_col)

def load_ieee(folder, files_dict, time_col):
    trans = pd.read_csv(os.path.join(BASE_PATH, folder, files_dict["transaction"]))
    ident = pd.read_csv(os.path.join(BASE_PATH, folder, files_dict["identity"]))
    df = trans.merge(ident, on="TransactionID", how="left")
    return df.sort_values(time_col)

def load_europeancard(folder, file_pattern, time_col):
    path = os.path.join(BASE_PATH, folder, file_pattern)
    df = pd.read_csv(path)
    return df.sort_values(time_col)

LOADERS = {
    "Sparkov": load_sparkov,
    "IEEE-CIS": load_ieee,
    "EuropeanCard": load_europeancard
}

In [5]:
def time_based_split(df, time_col, target_col):
    """Sort by time, then split 60‑20‑20."""
    df = df.sort_values(time_col).reset_index(drop=True)
    n = len(df)
    train_end = int(0.6 * n)
    val_end = int(0.8 * n)
    train = df.iloc[:train_end].copy()
    val   = df.iloc[train_end:val_end].copy()
    test  = df.iloc[val_end:].copy()
    X_train, y_train = train.drop(columns=[target_col]), train[target_col]
    X_val, y_val     = val.drop(columns=[target_col]),   val[target_col]
    X_test, y_test   = test.drop(columns=[target_col]),  test[target_col]
    return X_train, y_train, X_val, y_val, X_test, y_test

In [6]:
def detect_types(X):
    """Return list of categorical and numerical columns."""
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    # Sometimes integer columns with few unique values are categorical
    int_cols = X.select_dtypes(include=['int']).columns
    for c in int_cols:
        if c not in cat_cols and X[c].nunique() < 10:
            cat_cols.append(c)
            num_cols.remove(c)
    return cat_cols, num_cols

In [7]:
def frequency_encode_high_cardinality(X_train, X_val, X_test, cat_cols, threshold=50):
    """
    For categorical columns with > `threshold` unique values,
    replace them with training‑set frequency and return the frequency map.
    Returns: X_train, X_val, X_test, low_card_cols, freq_maps
    """
    X_train = X_train.copy()
    X_val   = X_val.copy()
    X_test  = X_test.copy()
    freq_maps = {}

    high_card_cols = [col for col in cat_cols if X_train[col].nunique() > threshold]
    low_card_cols  = [col for col in cat_cols if col not in high_card_cols]

    for col in high_card_cols:
        freq_map = X_train[col].value_counts(normalize=True).to_dict()
        freq_maps[col] = freq_map
        X_train[col + '_freq'] = X_train[col].map(freq_map).fillna(0)
        X_val[col + '_freq']   = X_val[col].map(freq_map).fillna(0)
        X_test[col + '_freq']  = X_test[col].map(freq_map).fillna(0)
        X_train.drop(columns=col, inplace=True)
        X_val.drop(columns=col, inplace=True)
        X_test.drop(columns=col, inplace=True)

    return X_train, X_val, X_test, low_card_cols, freq_maps

In [8]:
def engineer_features(X_train, X_val, X_test, dataset_name):
    """
    Add domain‑specific features. All aggregations are computed from X_train
    only and then mapped to validation/test. Returns the transformed DataFrames
    and a dictionary of aggregation tables that must be stored for streaming use.
    """
    X_train = X_train.copy()
    X_val   = X_val.copy()
    X_test  = X_test.copy()
    agg_stats = {}          # will hold DataFrames like card_stats, merch_stats, etc.

    # ====================================================
    # 1. Sparkov
    # ====================================================
    if dataset_name == "Sparkov":
        # ----- datetime features -----
        if "unix_time" in X_train.columns:
            for X in [X_train, X_val, X_test]:
                X["trans_datetime"] = pd.to_datetime(X["unix_time"], unit='s')
                X["hour"] = X["trans_datetime"].dt.hour
                X["dayofweek"] = X["trans_datetime"].dt.dayofweek
                X["is_weekend"] = (X["dayofweek"] >= 5).astype(int)
                X["month"] = X["trans_datetime"].dt.month
                X.drop(columns=["trans_datetime"], inplace=True)

        # ----- log amount -----
        if "amt" in X_train.columns:
            for X in [X_train, X_val, X_test]:
                X["log_amt"] = np.log1p(X["amt"])

        # ----- per‑card aggregations -----
        if "cc_num" in X_train.columns:
            card_groups = X_train.groupby("cc_num")
            card_stats = pd.DataFrame({
                "card_txn_count":        card_groups.size(),
                "card_amt_mean":         card_groups["amt"].mean(),
                "card_amt_std":          card_groups["amt"].std().fillna(0),
                "card_amt_max":          card_groups["amt"].max(),
                "card_amt_min":          card_groups["amt"].min(),
                "card_unique_merchants": card_groups["merchant"].nunique(),
                "card_unique_categories": card_groups["category"].nunique(),
            }).reset_index()
            agg_stats["sparkov_card_stats"] = card_stats
            X_train = X_train.merge(card_stats, on="cc_num", how="left")
            X_val   = X_val.merge(card_stats, on="cc_num", how="left")
            X_test  = X_test.merge(card_stats, on="cc_num", how="left")

        # ----- per‑merchant aggregations -----
        if "merchant" in X_train.columns:
            merch_groups = X_train.groupby("merchant")
            merch_stats = pd.DataFrame({
                "merch_txn_count":    merch_groups.size(),
                "merch_amt_mean":     merch_groups["amt"].mean(),
                "merch_unique_cards": merch_groups["cc_num"].nunique(),
            }).reset_index()
            agg_stats["sparkov_merch_stats"] = merch_stats
            X_train = X_train.merge(merch_stats, on="merchant", how="left")
            X_val   = X_val.merge(merch_stats, on="merchant", how="left")
            X_test  = X_test.merge(merch_stats, on="merchant", how="left")

        # ----- customer‑merchant distance -----
        if "merch_lat" in X_train.columns and "merch_long" in X_train.columns:
            def haversine_vectorised(lat1, lon1, lat2, lon2):
                """
                Vectorised Haversine distance in kilometres.
                Works on numpy arrays of any shape.
                """
                lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
                dlat = lat2 - lat1
                dlon = lon2 - lon1
                a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
                c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
                return 6371.0 * c

            for X in [X_train, X_val, X_test]:
                X["customer_merchant_dist"] = haversine_vectorised(
                    X["lat"].values, X["long"].values,
                    X["merch_lat"].values, X["merch_long"].values
                )

    # ====================================================
    # 2. IEEE‑CIS
    # ====================================================
    elif dataset_name == "IEEE-CIS":
        # ----- synthetic datetime -----
        if "TransactionDT" in X_train.columns:
            start_date = pd.Timestamp("2017-12-01")
            for X in [X_train, X_val, X_test]:
                X["DT"] = start_date + pd.to_timedelta(X["TransactionDT"], unit='s')
                X["hour"] = X["DT"].dt.hour
                X["dayofweek"] = X["DT"].dt.dayofweek
                X["is_weekend"] = (X["dayofweek"] >= 5).astype(int)
                X["month"] = X["DT"].dt.month
                X.drop(columns=["DT"], inplace=True)

        # ----- log amount -----
        if "TransactionAmt" in X_train.columns:
            for X in [X_train, X_val, X_test]:
                X["log_TransactionAmt"] = np.log1p(X["TransactionAmt"])

        # ----- per‑card (card1) aggregations -----
        if "card1" in X_train.columns:
            card_groups = X_train.groupby("card1")
            card_stats = pd.DataFrame({
                "card1_txn_count":      card_groups.size(),
                "card1_amt_mean":       card_groups["TransactionAmt"].mean(),
                "card1_amt_std":        card_groups["TransactionAmt"].std().fillna(0),
                "card1_amt_max":        card_groups["TransactionAmt"].max(),
                "card1_amt_min":        card_groups["TransactionAmt"].min(),
                "card1_unique_product": card_groups["ProductCD"].nunique(),
            }).reset_index()
            agg_stats["ieee_card1_stats"] = card_stats
            X_train = X_train.merge(card_stats, on="card1", how="left")
            X_val   = X_val.merge(card_stats, on="card1", how="left")
            X_test  = X_test.merge(card_stats, on="card1", how="left")

        # ----- per‑billing address aggregations -----
        if "addr1" in X_train.columns:
            addr_groups = X_train.groupby("addr1")
            addr_stats = pd.DataFrame({
                "addr1_txn_count":      addr_groups.size(),
                "addr1_amt_mean":       addr_groups["TransactionAmt"].mean(),
                "addr1_unique_cards":   addr_groups["card1"].nunique(),
                "addr1_unique_product": addr_groups["ProductCD"].nunique(),
            }).reset_index()
            agg_stats["ieee_addr1_stats"] = addr_stats
            X_train = X_train.merge(addr_stats, on="addr1", how="left")
            X_val   = X_val.merge(addr_stats, on="addr1", how="left")
            X_test  = X_test.merge(addr_stats, on="addr1", how="left")

        # ----- email domain aggregations -----
        for col in ["P_emaildomain", "R_emaildomain"]:
            if col in X_train.columns:
                dom_groups = X_train.groupby(col)
                dom_stats = pd.DataFrame({
                    f"{col}_txn_count":   dom_groups.size(),
                    f"{col}_amt_mean":    dom_groups["TransactionAmt"].mean(),
                    f"{col}_unique_cards":dom_groups["card1"].nunique(),
                }).reset_index()
                agg_stats[f"ieee_{col}_stats"] = dom_stats
                X_train = X_train.merge(dom_stats, on=col, how="left")
                X_val   = X_val.merge(dom_stats, on=col, how="left")
                X_test  = X_test.merge(dom_stats, on=col, how="left")

    # ====================================================
    # 3. European Card Holders 2013
    # ====================================================
    elif dataset_name == "EuropeanCard":
        # ----- time features (seconds → hour, day, week) -----
        if "Time" in X_train.columns:
            for X in [X_train, X_val, X_test]:
                X["hour"] = (X["Time"] // 3600) % 24
                X["day"]  = X["Time"] // (24 * 3600)
                X["week"] = X["Time"] // (7 * 24 * 3600)

        # ----- log amount -----
        if "Amount" in X_train.columns:
            for X in [X_train, X_val, X_test]:
                X["log_Amount"] = np.log1p(X["Amount"])

        # (No aggregation features – dataset has no card/account identifiers)

    # ====================================================
    # Fill NaN from unseen entities with training‑set fallbacks
    # ====================================================
    # Collect names of all newly added columns (those that came from merges)
    # We take all numeric columns that still have NaN and fill them.
    for col in X_train.columns:
        if X_train[col].isnull().any():
            if np.issubdtype(X_train[col].dtype, np.number):
                fill_val = X_train[col].median()
            else:
                fill_val = 0
            X_train[col] = X_train[col].fillna(fill_val)
            X_val[col]   = X_val[col].fillna(fill_val)
            X_test[col]  = X_test[col].fillna(fill_val)

    # ====================================================
    # Drop high‑cardinality original ID columns
    # ====================================================
    drop_cols = ["cc_num", "merchant", "nameOrig", "nameDest", "trans_num",
                 "TransactionID", "card1", "addr1", "P_emaildomain", "R_emaildomain",
                 "DeviceInfo"]
    for col in drop_cols:
        for X in [X_train, X_val, X_test]:
            if col in X.columns:
                X.drop(columns=col, inplace=True)

    return X_train, X_val, X_test, agg_stats

In [9]:
def get_fs_methods(num_features=None):
    """
    Generate a list of (method_name, k, selector_object) for different k.
    Fast, scalable methods only:
      - ANOVA F‑test, Mutual Information (SelectKBest)
      - ANOVA SelectPercentile
      - VarianceThreshold (constant feature removal)
      - NoSelection baseline
    """
    methods = []
    possible_k = [5, 10, 15, 20, 30, 50, 'all']

    # Boundary check: drop integer k values larger than the actual feature count
    if num_features is not None:
        possible_k = [k for k in possible_k if k == 'all' or k <= num_features]

    for k in possible_k:
        # ----- ANOVA F‑test -----
        selector = SelectKBest(f_classif, k=k)
        methods.append((f'ANOVA_k{k}', k, selector))

        # ----- Mutual Information -----
        selector = SelectKBest(mutual_info_classif, k=k)
        methods.append((f'MI_k{k}', k, selector))

    # ----- Percentile‑based selection (fast filter) -----
    for perc in [10, 20, 30, 50]:
        selector = SelectPercentile(f_classif, percentile=perc)
        methods.append((f'ANOVA_Percentile{perc}', 'all', selector))

    # ----- Remove constant features (variance = 0) -----
    selector = VarianceThreshold(threshold=0)
    methods.append(('VarianceThreshold', 'all', selector))

    # ----- Baseline: keep all features -----
    methods.append(('NoSelection_all', 'all', None))

    return methods

In [10]:
"""
NearMiss-1: Selects majority class samples whose average distance to the k closest minority class samples is the smallest.
NearMiss-2: Selects majority class samples whose average distance to the k farthest minority class samples is the smallest.
NearMiss-3: A two-step algorithm. First, for each minority class sample, it finds its k nearest majority class neighbors. Then, it selects the majority class samples that maintain the largest average distance to those neighbors, keeping data points that sit on the boundary between classes.
"""

def create_sampler(name, y_train):
    """
    Return a sampler instance with hyperparameters automatically adapted
    to the minority class size in y_train.
    """
    minority_count = (y_train == 1).sum() if hasattr(y_train, 'sum') else sum(y_train)
    # Safe k_neighbors for k‑NN based samplers
    safe_k = max(1, min(5, minority_count - 1)) if minority_count > 1 else 1

    if name == 'SMOTE':
        return SMOTE(random_state=42, k_neighbors=safe_k)
    elif name == 'BorderlineSMOTE':
        return BorderlineSMOTE(random_state=42, k_neighbors=safe_k)
    elif name == 'ADASYN':
        return ADASYN(random_state=42, n_neighbors=safe_k)
    elif name == 'RandomUnderSampler':
        return RandomUnderSampler(random_state=42)
    elif name == 'ClusterCentroids':
        return ClusterCentroids(random_state=42)
    elif name == 'NearMiss':
        return NearMiss(version=3)
    elif name == 'TomekLinks':
        return TomekLinks()
    elif name == 'EditedNearestNeighbours':
        return EditedNearestNeighbours()
    elif name == 'SMOTEENN':
        return SMOTEENN(random_state=42,
                        smote=SMOTE(random_state=42, k_neighbors=safe_k))
    elif name == 'SMOTETomek':
        return SMOTETomek(random_state=42,
                          smote=SMOTE(random_state=42, k_neighbors=safe_k))
    else:
        raise ValueError(f"Unknown sampler name: {name}")

In [11]:
def process_dataset(dataset_name, config):
    print(f"\n{'='*50}\nProcessing {dataset_name}\n{'='*50}")
    loader = LOADERS[dataset_name]
    time_col = config['time_col']
    target_col = config['target_col']

    # Load and sample 
    df = loader(config['folder'], config['files'], time_col)

    # Standardize target column name to 'Fraud' for all datasets
    df = df.rename(columns={target_col: 'Fraud'})
    target_col = 'Fraud'
    
    #uncoment the next line for testing
    #df = df.sample(frac=TEST_FRACTION, random_state=42)   # 0.1% test sample

    # Temporal split
    X_train, y_train, X_val, y_val, X_test, y_test = time_based_split(
        df, time_col, target_col
    )

    val_out = X_val.copy()
    val_out[target_col] = y_val.values
    val_out.to_csv(os.path.join(OUTPUT_DIR, f"{dataset_name}_val.csv"), index=False)

    test_out = X_test.copy()
    test_out[target_col] = y_test.values
    test_out.to_csv(os.path.join(OUTPUT_DIR, f"{dataset_name}_test.csv"), index=False)
    
    # Missing values (imputers are created here)
    imp_cat = SimpleImputer(strategy='most_frequent')
    imp_num = SimpleImputer(strategy='median')

    missing_pct = X_train.isnull().mean()
    cols_to_drop = missing_pct[missing_pct > 0.05].index.tolist()
    X_train = X_train.drop(columns=cols_to_drop)
    X_val   = X_val.drop(columns=cols_to_drop)
    X_test  = X_test.drop(columns=cols_to_drop)

    cat_cols, num_cols = detect_types(X_train)

    if cat_cols:
        X_train[cat_cols] = imp_cat.fit_transform(X_train[cat_cols])
        X_val[cat_cols]   = imp_cat.transform(X_val[cat_cols])
        X_test[cat_cols]  = imp_cat.transform(X_test[cat_cols])
    if num_cols:
        X_train[num_cols] = imp_num.fit_transform(X_train[num_cols])
        X_val[num_cols]   = imp_num.transform(X_val[num_cols])
        X_test[num_cols]  = imp_num.transform(X_test[num_cols])

    # Feature engineering (now returns agg_stats)
    X_train, X_val, X_test, agg_stats = engineer_features(X_train, X_val, X_test, dataset_name)
    cat_cols, num_cols = detect_types(X_train)

    # Frequency encode high‑cardinality categoricals (returns freq_maps)
    X_train, X_val, X_test, cat_cols, freq_maps = frequency_encode_high_cardinality(
        X_train, X_val, X_test, cat_cols, threshold=50
    )
    cat_cols, num_cols = detect_types(X_train)

    # Encode & scale (save encoder and scaler)
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    scaler = StandardScaler()
    if cat_cols:
        X_train[cat_cols] = enc.fit_transform(X_train[cat_cols])
        X_val[cat_cols]   = enc.transform(X_val[cat_cols])
        X_test[cat_cols]  = enc.transform(X_test[cat_cols])
    if num_cols:
        X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
        X_val[num_cols]   = scaler.transform(X_val[num_cols])
        X_test[num_cols]  = scaler.transform(X_test[num_cols])

    # ----- Save base transforms, aggregation stats, and frequency maps -----
    base_state = {
        'imputer_cat': imp_cat,
        'imputer_num': imp_num,
        'encoder': enc,
        'scaler': scaler,
        'cat_cols': cat_cols,
        'num_cols': num_cols,
        'agg_stats': agg_stats,
        'freq_maps': freq_maps,
    }
    base_path = os.path.join(OUTPUT_DIR, f"{dataset_name}_base_state.joblib")
    joblib.dump(base_state, base_path)
    # --------------------------------------------------------------------

    fs_methods = get_fs_methods(num_features=X_train.shape[1])
    for fs_name, k, selector in fs_methods:
        # ----- Feature selection (timed) -----
        t_sel_start = time.time()
        if selector is None:
            X_tr_fs, X_val_fs, X_test_fs = X_train, X_val, X_test
        else:
            selector.fit(X_train, y_train)
            mask = selector.get_support()
            X_tr_fs = X_train.loc[:, mask]
            X_val_fs = X_val.loc[:, mask]
            X_test_fs = X_test.loc[:, mask]
        selection_time = time.time() - t_sel_start

        # Loop over sampling techniques
        sampler_names = [
            'SMOTE', 'BorderlineSMOTE', 'ADASYN', 'RandomUnderSampler',
            'ClusterCentroids', 'NearMiss', 'TomekLinks',
            'EditedNearestNeighbours', 'SMOTEENN', 'SMOTETomek'
        ]
        for samp_name in sampler_names:
            try:
                # ----- Sampling + saving (timed) -----
                t_samp_start = time.time()

                sampler = create_sampler(samp_name, y_train)
                X_tr_fs = X_tr_fs.astype(float).fillna(0.0)
                X_tr_res, y_tr_res = sampler.fit_resample(X_tr_fs, y_train)
                train_res_df = pd.DataFrame(X_tr_res, columns=X_tr_fs.columns)
                train_res_df[target_col] = y_tr_res

                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                fname = f"{samp_name}--{fs_name}--{dataset_name}--{timestamp}.csv"
                path = os.path.join(OUTPUT_DIR, fname)
                train_res_df.to_csv(path, index=False)

                # Save combo state
                combo_state = {}
                if selector is not None:
                    combo_state['selector'] = selector
                combo_state['sampler'] = sampler
                state_fname = f"{samp_name}--{fs_name}--{dataset_name}--{timestamp}_state.joblib"
                joblib.dump(combo_state, os.path.join(OUTPUT_DIR, state_fname))

                sampling_time = time.time() - t_samp_start
                total_time = selection_time + sampling_time

                # ----- Count legit/fraud -----
                legit_count = (y_tr_res == 0).sum()
                fraud_count = (y_tr_res == 1).sum()

                # ----- Log -----
                processing_log.append({
                    'file_name': fname,
                    'dataset': dataset_name,
                    'fs_method': fs_name,
                    'sampler': samp_name,
                    'legitimate': legit_count,
                    'fraud': fraud_count,
                    'selection_time_sec': round(selection_time, 4),
                    'sampling_time_sec': round(sampling_time, 4),
                    'total_time_sec': round(total_time, 4)
                })

                print(f"Saved: {fname}  (selection: {selection_time:.2f}s, sampling: {sampling_time:.2f}s, total: {total_time:.2f}s)")
            except Exception as e:
                print(f"Skipped {samp_name} + {fs_name}: {str(e)[:100]}")
                continue

In [12]:
for ds_name, ds_cfg in DATASETS.items():
    process_dataset(ds_name, ds_cfg)


Processing Sparkov
Saved: SMOTE--ANOVA_k5--Sparkov--20260628_184524.csv  (selection: 0.29s, sampling: 7.67s, total: 7.96s)
Saved: BorderlineSMOTE--ANOVA_k5--Sparkov--20260628_184533.csv  (selection: 0.29s, sampling: 8.52s, total: 8.82s)
Saved: ADASYN--ANOVA_k5--Sparkov--20260628_184541.csv  (selection: 0.29s, sampling: 8.45s, total: 8.74s)
Saved: RandomUnderSampler--ANOVA_k5--Sparkov--20260628_184549.csv  (selection: 0.29s, sampling: 0.15s, total: 0.45s)
Saved: ClusterCentroids--ANOVA_k5--Sparkov--20260628_190325.csv  (selection: 0.29s, sampling: 1055.80s, total: 1056.09s)
Saved: NearMiss--ANOVA_k5--Sparkov--20260628_190326.csv  (selection: 0.29s, sampling: 1.09s, total: 1.38s)
Saved: TomekLinks--ANOVA_k5--Sparkov--20260628_190330.csv  (selection: 0.29s, sampling: 8.32s, total: 8.61s)
Saved: EditedNearestNeighbours--ANOVA_k5--Sparkov--20260628_190339.csv  (selection: 0.29s, sampling: 8.70s, total: 8.99s)
Saved: SMOTEENN--ANOVA_k5--Sparkov--20260628_190353.csv  (selection: 0.29s, sampl

In [13]:
# Save processing log
log_df = pd.DataFrame(processing_log)
log_path = "processing_times.csv"
log_df.to_csv(log_path, index=False)
print(f"\nProcessing log saved to {log_path}")
print(f"Total combinations logged: {len(log_df)}")


Processing log saved to processing_times.csv
Total combinations logged: 560
